In [1]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")

Using openai/gpt-oss-120b on Groq


In [7]:
# os.environ["TAVILY_API_KEY"] = "tvly-..."
# os.environ["ALPHA_VANTAGE_API_KEY"] = "..."

if not os.getenv("TAVILY_API_KEY"):
    print("⚠️  TAVILY_API_KEY not set — the news and sentiment agents cannot search the web.")
else:
    print("✓ TAVILY_API_KEY is configured")

if not os.getenv("ALPHA_VANTAGE_API_KEY"):
    print("ℹ️  ALPHA_VANTAGE_API_KEY not set — financial data comes from yfinance (free, no key).")
else:
    print("✓ ALPHA_VANTAGE_API_KEY is configured")

✓ TAVILY_API_KEY is configured
✓ ALPHA_VANTAGE_API_KEY is configured


In [8]:
from strands import Agent
from strands.multiagent import GraphBuilder

In [9]:
import yfinance as yf
from strands import Agent, ToolContext, tool
from strands_tools.tavily import tavily_search


# Groq's free tier rejects any single request over 8,000 tokens ("Request too large").
# A default Tavily search returns ~3,700 tokens, so a worker that searches twice overflows.
# The workers use this wrapper instead: the same Tavily search, top 3 results (~1,100 tokens),
# at most twice per task, because every result is resent with the worker's next request.
@tool(context=True)
async def web_search(query: str, tool_context: ToolContext, time_range: str = "month") -> dict | str:
    """Search the web with Tavily and return the top 3 results.

    Args:
        query: What to search for
        time_range: How recent the results should be: day, week, month or year
    """
    searches = tool_context.agent.state.get("searches") or 0  # as_tool() resets state for each task
    if searches >= 2:
        return "Search limit reached: answer with the results you already have."
    tool_context.agent.state.set("searches", searches + 1)
    return await tavily_search(query=query, max_results=3, time_range=time_range)